# 0 — Markdown Extraction

Extract markdown from **all PDFs** in `email/attachements/` using `prebuilt-layout`.
Output is saved to `output/markdown/{stem}.md` for downstream analysis of table
patterns and financial statement structures.

In [1]:
import json, os, re, time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

endpoint = os.environ["FOUNDRY_ENDPOINT"].split("/api/projects/")[0].rstrip("/") + "/"
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"],
)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version="2025-11-01")
#print("Client ready:", endpoint)

In [2]:
# Discover PDFs and set up output directory
pdf_dir = REPO_ROOT / "email" / "attachements"
pdfs = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs:")
for p in pdfs:
    print(f"  {p.name}")

md_dir = REPO_ROOT / "output" / "markdown"
md_dir.mkdir(parents=True, exist_ok=True)
print(f"\nOutput dir: {md_dir}")

Found 5 PDFs:
  10K-GOOG-01-31-2024.pdf
  10K-MSFT-07-27-2023.pdf
  bnl_10K_2024.pdf
  Safehold 2024 10-K (1) 1.pdf
  trno_10Q_Q3_2024.pdf

Output dir: c:\Users\jomedin\Documents\SECExtraction\output\markdown


In [3]:
# Extract markdown from each PDF using prebuilt-layout
for pdf in pdfs:
    md_path = md_dir / f"{pdf.stem}.md"
    if md_path.exists():
        print(f"  SKIP {pdf.name} (already exists)")
        continue

    print(f"  Extracting {pdf.name} ...", end=" ", flush=True)
    t0 = time.time()
    poller = client.begin_analyze_binary(
        analyzer_id="prebuilt-layout",
        binary_input=pdf.read_bytes(),
        content_type="application/pdf",
    )
    result = poller.result().as_dict()
    markdown = result["contents"][0]["markdown"]
    md_path.write_text(markdown, encoding="utf-8")
    elapsed = time.time() - t0
    n_pages = result["contents"][0].get("endPageNumber", "?")
    print(f"{n_pages} pages, {len(markdown):,} chars, {elapsed:.1f}s")

print("\nDone.")

  SKIP 10K-GOOG-01-31-2024.pdf (already exists)
  SKIP 10K-MSFT-07-27-2023.pdf (already exists)
  SKIP bnl_10K_2024.pdf (already exists)
  SKIP Safehold 2024 10-K (1) 1.pdf (already exists)
  SKIP trno_10Q_Q3_2024.pdf (already exists)

Done.


In [4]:
# Summary of all extracted markdown files
for md_path in sorted(md_dir.glob("*.md")):
    text = md_path.read_text(encoding="utf-8")
    tables = re.findall(r"<table>.*?</table>", text, re.DOTALL)
    page_breaks = len(re.findall(r"<!-- PageBreak -->", text))
    n_pages = page_breaks + 1  # N breaks = N+1 pages
    print(f"{md_path.name}: {len(text):>8,} chars, {n_pages} pages, {len(tables)} HTML tables")

10K-GOOG-01-31-2024.md:  440,353 chars, 109 pages, 71 HTML tables
10K-MSFT-07-27-2023.md:  414,700 chars, 116 pages, 82 HTML tables
bnl_10K_2024.md:  562,751 chars, 118 pages, 75 HTML tables
Safehold 2024 10-K (1) 1.md:  496,752 chars, 113 pages, 64 HTML tables
trno_10Q_Q3_2024.md:  205,026 chars, 51 pages, 45 HTML tables


## Financial Statement Table Analysis

Search the markdown for financial statement sections and catalogue the table patterns:
- How each statement appears (single page vs multi-page)
- Table header structures (single-level vs two-level spanning headers)
- Multi-page continuation patterns (with/without repeated headers)

In [5]:
# Shared FS-title patterns used by the detection and analysis cells below
# Order matters: more specific patterns first (ComprehensiveIncome before IncomeStatement)
# Two-phase detection: 1) caption text, 2) fallback to pre-table context
_FS_PATTERNS = [
    ("BalanceSheet", r"(?:Consolidated\s+)?Balance\s+Sheet"),
    ("ComprehensiveIncome", r"(?:Consolidated\s+)?(?:Statements?\s+of\s+)?Comprehensive\s+(?:Income|Loss)\s+Statement"),
    ("ComprehensiveIncome", r"(?:Consolidated\s+)?Comprehensive\s+Income\s+Statement"),
    ("ComprehensiveIncome", r"(?:Consolidated\s+)?Statements?\s+of\s+Comprehensive\s+(?:Income|Loss)"),
    ("ComprehensiveIncome", r"Income\s+and\s+Comprehensive\s+(?:Income|Loss)"),
    ("IncomeStatement", r"(?:Consolidated\s+)?(?:Statements?\s+of\s+)?(?:Operations|(?:Net\s+)?Income)(?:\s+Statement)?(?:\s*\(|(?:\s+and))"),
    ("IncomeStatement", r"(?:Consolidated\s+)?Statements?\s+of\s+(?:Operations|(?:Net\s+)?Income)"),
    ("IncomeStatement", r"Income\s+Statements?\s*\("),
    ("Equity", r"(?:Consolidated\s+)?Statements?\s+of\s+(?:Changes\s+in\s+)?(?:Stockholders.{0,5}|Shareholders.{0,5})?Equity"),
    ("Equity", r"Stockholders.{0,5}\s*Equity\s+Statements?"),
    ("CashFlow", r"(?:Consolidated\s+)?Statements?\s+of\s+Cash\s+Flows?"),
    ("CashFlow", r"Cash\s+Flows?\s+Statements?"),
]

def _classify_fs(text: str) -> str:
    """Classify a text string as a financial statement type (first match)."""
    for ft, pat in _FS_PATTERNS:
        if re.search(pat, text, re.IGNORECASE):
            return ft
    return ""

def _classify_fs_all(text: str) -> list[str]:
    """Return all distinct FS types that match the text.
    Handles combined statements like 'Income and Comprehensive Income'.
    Avoids false IncomeStatement matches on pure ComprehensiveIncome titles."""
    seen = set()
    result = []
    for ft, pat in _FS_PATTERNS:
        if ft not in seen and re.search(pat, text, re.IGNORECASE):
            seen.add(ft)
            result.append(ft)
    # Remove false IncomeStatement when it only matched via "Comprehensive Income"
    if "IncomeStatement" in result and "ComprehensiveIncome" in result:
        # Keep IS only if there's an IS-specific match NOT preceded by "Comprehensive"
        has_real_is = bool(re.search(
            r"(?<!Comprehensive\s)(?:Statements?\s+of\s+(?:Operations|(?:Net\s+)?Income)|Income\s+Statements?\s*\()",
            text, re.IGNORECASE
        ))
        # Also keep IS if title explicitly says "Income and Comprehensive"
        has_combined = bool(re.search(r"Income\s+and\s+Comprehensive", text, re.IGNORECASE))
        if not has_real_is and not has_combined:
            result.remove("IncomeStatement")
    return result

def _page_at(md_text: str, pos: int) -> int:
    """Return the page number (1-based) at a given character offset."""
    return 1 + len(re.findall(r"<!-- PageBreak -->", md_text[:pos]))

def _extract_context_text(pre: str) -> str:
    """Extract FS-relevant text from pre-table context.
    Combines bold (**...**), heading (#), and all-caps lines."""
    bold_text = " ".join(re.findall(r"\*\*(.*?)\*\*", pre))
    heading_text = " ".join(re.findall(r"^#+\s+(.+)", pre, re.MULTILINE))
    allcaps_text = " ".join(
        line.strip() for line in pre.splitlines()
        if line.strip() and re.match(r"^[A-Z\s',()]+$", line.strip()) and len(re.findall(r"[A-Z]", line)) >= 3
    )
    return f"{bold_text} {heading_text} {allcaps_text}"

def _get_nearby_context(md_text: str, table_start: int, lookback: int = 1500) -> str:
    """Get pre-table context limited to current page + 1 previous page.
    Strips embedded tables, then trims to at most 1 PageBreak of distance."""
    pre_start = max(0, table_start - lookback)
    pre = md_text[pre_start:table_start]
    pre = re.sub(r"<table>.*?</table>", "", pre, flags=re.DOTALL)
    breaks = [m.end() for m in re.finditer(r"<!-- PageBreak -->", pre)]
    if len(breaks) >= 2:
        pre = pre[breaks[-2]:]
    return pre

def classify_tables(md_text: str):
    """Classify all tables in a markdown document.
    Returns list of (table_idx, page, fs_type, n_hdr, n_data, caption).
    Combined statements (e.g. 'Income and Comprehensive Income') produce
    multiple entries for the same table — one per FS type."""
    all_tables = list(re.finditer(r"<table>(.*?)</table>", md_text, re.DOTALL))
    results = []
    for i, tm in enumerate(all_tables):
        html = tm.group(0)
        page = _page_at(md_text, tm.start())
        hdr_rows = re.findall(r"<tr>\s*<th.*?</tr>", html, re.DOTALL)
        dat_rows = re.findall(r"<tr>\s*<td.*?</tr>", html, re.DOTALL)

        cap_m = re.search(r"<caption>(.*?)</caption>", html, re.DOTALL)
        caption = re.sub(r"<[^>]+>", "", cap_m.group(1)).strip() if cap_m else ""
        fs_types = _classify_fs_all(caption) if caption else []

        if not fs_types:
            pre = _get_nearby_context(md_text, tm.start())
            ctx = _extract_context_text(pre) + " " + caption
            fs_types = _classify_fs_all(ctx)

        if fs_types:
            for ft in fs_types:
                results.append((i, page, ft, len(hdr_rows), len(dat_rows), caption[:80]))
        else:
            results.append((i, page, "", len(hdr_rows), len(dat_rows), caption[:80]))
    return results

print("classify_tables() defined — with _classify_fs_all for combined statements")

classify_tables() defined — with _classify_fs_all for combined statements


In [6]:
# Detect financial statement sections in each markdown file
for md_path in sorted(md_dir.glob("*.md")):
    text = md_path.read_text(encoding="utf-8")
    print(f"\n{'='*80}")
    print(f"{md_path.stem}")
    print(f"{'='*80}")

    for fs_type, pattern in _FS_PATTERNS:
        matches = list(re.finditer(pattern, text, re.IGNORECASE))
        if not matches:
            continue
        print(f"\n  {fs_type}: {len(matches)} occurrence(s)")
        for m in matches:
            page = _page_at(text, m.start())
            start = max(0, m.start() - 30)
            end = min(len(text), m.end() + 80)
            ctx = text[start:end].replace("\n", " ").strip()
            print(f"    Page {page}: ...{ctx[:140]}...")


10K-GOOG-01-31-2024

  BalanceSheet: 23 occurrence(s)
    Page 43: ...t are not yet recorded on our Consolidated Balance Sheets. These leases will commence between 2024 and 2026 with non-cancelable lease te...
    Page 44: ...er current liabilities on our Consolidated Balance Sheets as we provided bank guarantees (in lieu of a cash payment) for the fines. For...
    Page 46: ...functional currencies at the balance sheet date, it would have resulted in an adverse effect on income before income taxes...
    Page 47: ...current fair value as of the balance sheet date. These investments, especially those that are in the early stages, are inh...
    Page 48: .../td> <td></td> </tr> <tr> <td>Consolidated Balance Sheets</td> <td>51</td> </tr> <tr> <td>Consolidated Statements of Income</td> <td>52<...
    Page 49: ...have audited the accompanying consolidated balance sheets of Alphabet Inc. (the Company) as of December 31, 2022 and 2023, the related c...
    Page 52: ...table> <caption>Alphab

## Multi-page Table Patterns

Identify which tables span multiple pages and how they continue:
- **Pattern A**: Same section, multiple tables with different headers (e.g., Equity 2024 + 2023)
- **Pattern B**: Table cut mid-row, continues on next page **without** headers
- **Pattern C**: Table cut, continues on next page **with** repeated headers

In [8]:
# Multi-page table pattern detection
def _get_headers(html):
    hdr_rows = re.findall(r"<tr>\\s*<th.*?</tr>", html, re.DOTALL)
    result = []
    for hr in hdr_rows:
        cells = re.findall(r"<th[^>]*>(.*?)</th>", hr, re.DOTALL)
        cells = [re.sub(r"<[^>]+>", "", c).strip() for c in cells]
        result.append(cells)
    return result

def _first_label(html):
    drs = re.findall(r"<tr>\\s*<td.*?</tr>", html, re.DOTALL)
    if drs:
        m = re.search(r"<td[^>]*>(.*?)</td>", drs[0], re.DOTALL)
        if m:
            return re.sub(r"<[^>]+>", "", m.group(1)).strip()[:80]
    return ""

for md_path in sorted(md_dir.glob("*.md")):
    text = md_path.read_text(encoding="utf-8")
    print(f"\n{'='*80}")
    print(f"{md_path.stem} — Multi-page Pattern Detection")
    print(f"{'='*80}")

    tables = list(re.finditer(r"<table>(.*?)</table>", text, re.DOTALL))

    for i in range(len(tables) - 1):
        t1, t2 = tables[i], tables[i + 1]
        p1 = _page_at(text, t1.start())
        p2 = _page_at(text, t2.start())

        if p2 - p1 > 1: continue

        h1 = _get_headers(t1.group(0))
        h2 = _get_headers(t2.group(0))
        fl1 = _first_label(t1.group(0))
        fl2 = _first_label(t2.group(0))

        pre1 = text[max(0, t1.start()-500):t1.start()]
        pre2 = text[max(0, t2.start()-500):t2.start()]
        fs1 = fs2 = ""
        for ft, pat in _FS_PATTERNS:
            if re.search(pat, pre1, re.IGNORECASE): fs1 = ft
            if re.search(pat, pre2, re.IGNORECASE): fs2 = ft

        between = re.sub(r"<!--.*?-->", "", text[t1.end():t2.start()]).strip()

        if h1 and h2 and h1 == h2 and p2 > p1:
            pattern = "C: continuation WITH repeated headers"
        elif h2 and h1 and h1 != h2 and fs1 == fs2 and fs1:
            pattern = f"A: same section ({fs1}), different sub-table headers"
        elif not h2 and p2 > p1:
            pattern = "B: continuation WITHOUT headers"
        elif p1 == p2 and fs1 == fs2 and fs1 and h1 != h2:
            pattern = f"A: same page ({fs1}), separate sub-tables"
        else: continue

        print(f"\n  Tables {i+1} -> {i+2} | Pages {p1} -> {p2} | {pattern}")
        if fs1 or fs2: print(f"    FS type: {fs1} -> {fs2}")
        print(f"    T{i+1} headers: {h1[:2]}")
        print(f"    T{i+2} headers: {h2[:2]}")
        print(f"    T{i+1} first row: {fl1}")
        print(f"    T{i+2} first row: {fl2}")
        if between: print(f"    Between: {between[:150]}")


10K-GOOG-01-31-2024 — Multi-page Pattern Detection

  Tables 3 -> 4 | Pages 35 -> 36 | B: continuation WITHOUT headers
    T3 headers: []
    T4 headers: []
    T3 first row: 
    T4 first row: 
    Between: NM = Not Meaningful

(1) See "Use of Non-GAAP Constant Currency Information" below for details relating to our use of constant currency information.



  Tables 4 -> 5 | Pages 36 -> 37 | B: continuation WITHOUT headers
    T4 headers: []
    T5 headers: []
    T4 first row: 
    T5 first row: 
    Between: #### Google Services


##### Google advertising revenues


##### Google Search & other

Google Search & other revenues increased $12.6 billion from 20

  Tables 6 -> 7 | Pages 37 -> 38 | B: continuation WITHOUT headers
    T6 headers: []
    T7 headers: []
    T6 first row: 
    T7 first row: 
    Between: For additional information, see Note 2 of the Notes to Consolidated Financial Statements included in Item 8 of this
Annual Report on Form 10-K.








  Tables 7 -> 8 | Pages

In [9]:
# Financial Statement Table Catalogue
# Two tiers: (1) caption-matched = high confidence, (2) context-matched = needs review
# Combined statements (e.g. "Income and Comprehensive Income") emit one entry per type
print("=" * 80)
print("FINANCIAL STATEMENT TABLE CATALOGUE")
print("=" * 80)

_all_fs_tables = {}

for md_path in sorted(md_dir.glob("*.md")):
    text = md_path.read_text(encoding="utf-8")
    all_tables = list(re.finditer(r"<table>(.*?)</table>", text, re.DOTALL))

    caption_matched = []
    context_matched = []

    for i, tm in enumerate(all_tables):
        html = tm.group(0)
        page = _page_at(text, tm.start())
        hdr_rows = re.findall(r"<tr>\s*<th.*?</tr>", html, re.DOTALL)
        dat_rows = re.findall(r"<tr>\s*<td.*?</tr>", html, re.DOTALL)

        cap_m = re.search(r"<caption>(.*?)</caption>", html, re.DOTALL)
        caption = re.sub(r"<[^>]+>", "", cap_m.group(1)).strip() if cap_m else ""
        fs_types = _classify_fs_all(caption) if caption else []

        if fs_types:
            for ft in fs_types:
                caption_matched.append((i, page, ft, len(hdr_rows), len(dat_rows), caption[:80]))
        else:
            pre = _get_nearby_context(text, tm.start())
            ctx = _extract_context_text(pre) + " " + caption
            fs_types = _classify_fs_all(ctx)
            for ft in fs_types:
                context_matched.append((i, page, ft, len(hdr_rows), len(dat_rows), caption[:80]))

    _all_fs_tables[md_path.stem] = caption_matched + context_matched

    print(f"\n--- {md_path.stem} ---")
    print(f"  Caption-matched ({len(caption_matched)}):")
    for idx, pg, ft, nh, nd, cap in caption_matched:
        print(f"    T{idx+1:2d} pg{pg:3d} {ft:<20s} {nh}hdr {nd:3d}data  \"{cap[:55]}\"")
    if context_matched:
        print(f"  Context-matched ({len(context_matched)}):")
        for idx, pg, ft, nh, nd, cap in context_matched:
            cap_str = f'"{cap[:55]}"' if cap else "(no caption)"
            print(f"    T{idx+1:2d} pg{pg:3d} {ft:<20s} {nh}hdr {nd:3d}data  {cap_str}")

    all_fs = sorted(caption_matched + context_matched, key=lambda t: t[0])
    for j in range(len(all_fs) - 1):
        ft1, ft2 = all_fs[j][2], all_fs[j+1][2]
        pg1, pg2 = all_fs[j][1], all_fs[j+1][1]
        if ft1 == ft2 and abs(pg2 - pg1) <= 2:
            print(f"  >> MULTI-PAGE {ft1}: T{all_fs[j][0]+1}->T{all_fs[j+1][0]+1} pg{pg1}->{pg2}")

print("\n" + "=" * 80)
print("COVERAGE")
print("=" * 80)
expected = ["BalanceSheet", "IncomeStatement", "ComprehensiveIncome", "Equity", "CashFlow"]
for fname, tables in _all_fs_tables.items():
    found = {t[2] for t in tables}
    status = " | ".join(f"{'✓' if ft in found else '✗'} {ft}" for ft in expected)
    print(f"  {fname}: {status}")

FINANCIAL STATEMENT TABLE CATALOGUE

--- 10K-GOOG-01-31-2024 ---
  Caption-matched (5):
    T19 pg 52 BalanceSheet         2hdr  38data  "Alphabet Inc. CONSOLIDATED BALANCE SHEETS (in millions,"
    T20 pg 53 IncomeStatement      2hdr  15data  "Alphabet Inc. CONSOLIDATED STATEMENTS OF INCOME (in mil"
    T21 pg 54 ComprehensiveIncome  2hdr  13data  "Alphabet Inc. CONSOLIDATED STATEMENTS OF COMPREHENSIVE "
    T22 pg 55 Equity               2hdr  24data  "Alphabet Inc. CONSOLIDATED STATEMENTS OF STOCKHOLDERS' "
    T23 pg 56 CashFlow             2hdr  40data  "Alphabet Inc. CONSOLIDATED STATEMENTS OF CASH FLOWS (in"
  Context-matched (3):
    T13 pg 41 IncomeStatement      2hdr   9data  (no caption)
    T48 pg 76 IncomeStatement      1hdr  16data  (no caption)
    T50 pg 77 IncomeStatement      2hdr   9data  (no caption)
  >> MULTI-PAGE IncomeStatement: T48->T50 pg76->77

--- 10K-MSFT-07-27-2023 ---
  Caption-matched (4):
    T18 pg 58 IncomeStatement      1hdr  23data  "ITEM 8. FINANCI